# 🏭 SmartStock – Inventory Health & Turnover Analytics
### Stock Movement, Low Stock Deficits, Dead Stock & Turnover Ratios
**Objective**: Monitor warehouse stock levels, stockout risks, dormant inventory, and calculate inventory turnover ratios.


## 1. Load Inventory & Catalog Data


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_DIR = os.path.join('..', 'datasets')
df_products = pd.read_csv(os.path.join(DATA_DIR, 'products.csv'))
df_inv = pd.read_csv(os.path.join(DATA_DIR, 'inventory.csv'))
df_items = pd.read_csv(os.path.join(DATA_DIR, 'sale_items.csv'))

df_inv['created_at'] = pd.to_datetime(df_inv['created_at'])

print(f"Loaded {len(df_products)} products and {len(df_inv)} inventory movement logs.")
df_products.head(3)


## 2. Stock Valuation & Category Inventory Breakdown
Compute total capital tied up in inventory across categories.


In [ ]:
total_stock_value = (df_products['current_stock'] * df_products['purchase_price']).sum()
total_units_on_hand = df_products['current_stock'].sum()

print("=" * 55)
print(f"🏷️  Total Products in Catalog:    {len(df_products)}")
print(f"📦 Total Units in Stock:          {total_units_on_hand:,}")
print(f"💵 Total Inventory Value (Cost):  ${total_stock_value:,.2f}")
print("=" * 55)

cat_inv = df_products.groupby('category_name').agg(
    product_count=('product_id', 'count'),
    units_in_stock=('current_stock', 'sum'),
    stock_value=('inventory_value', 'sum')
).reset_index().sort_values(by='stock_value', ascending=False)

plt.figure(figsize=(12, 5))
plt.barh(cat_inv['category_name'][::-1], cat_inv['stock_value'][::-1], color='#3b82f6')
plt.title('Capital Tied Up by Product Category (Valuation at Cost)', fontweight='bold', fontsize=13)
plt.xlabel('Inventory Value ($)')
plt.gca().xaxis.set_major_formatter('${x:,.0f}')
plt.tight_layout()
plt.show()

display(cat_inv)


## 3. Stock Movement & Velocity Analysis
Classify transaction types: Inbound Replenishments vs. Outbound POS Sales.


In [ ]:
txn_counts = df_inv['type'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Transaction distribution
ax1.pie(txn_counts, labels=txn_counts.index, autopct='%1.1f%%', colors=['#10b981', '#3b82f6', '#f59e0b', '#ef4444'])
ax1.set_title('Inventory Transaction Distribution by Type', fontweight='bold')

# Net unit flow by category
inbound = df_inv[df_inv['quantity'] > 0].groupby('category_name')['quantity'].sum()
outbound = df_inv[df_inv['quantity'] < 0].groupby('category_name')['quantity'].sum().abs()
flow_df = pd.DataFrame({'Replenished (In)': inbound, 'Sold / Depleted (Out)': outbound}).fillna(0)

flow_df.plot(kind='bar', ax=ax2, colormap='tab10', alpha=0.85)
ax2.set_title('Stock Inflow vs Outflow by Category', fontweight='bold')
ax2.set_ylabel('Total Units')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## 4. Low Stock Deficit & Safety Buffer Alerts
Flag SKUs whose current stock level is below or dangerously close to their defined reorder point.


In [ ]:
df_products['deficit'] = df_products['reorder_level'] - df_products['current_stock']
low_stock_items = df_products[df_products['current_stock'] <= df_products['reorder_level']].sort_values(by='current_stock')

print(f"⚠️ {len(low_stock_items)} items are currently AT OR BELOW their reorder threshold!")

if len(low_stock_items) > 0:
    display(low_stock_items[['sku', 'name', 'category_name', 'current_stock', 'reorder_level', 'unit']])
else:
    print("All items currently satisfy minimum safety buffers.")


## 5. Dead Stock Identification & Inventory Turnover Ratio
Identify SKUs with stagnant velocity (zero sales in the period) and calculate annual inventory turnover ratio:
$$\text{Turnover Ratio} = \frac{\text{Total COGS}}{\text{Average Inventory Value}}$$
$$\text{Days Sales of Inventory (DSI)} = \frac{180}{\text{Turnover Ratio}}$$


In [ ]:
sold_prod_ids = set(df_items['product_id'].unique())
dead_stock = df_products[~df_products['product_id'].isin(sold_prod_ids)]
print(f"🛑 Dead Stock Products (0 Units Sold): {len(dead_stock)}")

# Compute Inventory Turnover Ratio
total_cogs = df_items['cogs'].sum()
avg_inv_value = total_stock_value
turnover_ratio = (total_cogs / avg_inv_value) if avg_inv_value > 0 else 0.0
dsi = (180 / turnover_ratio) if turnover_ratio > 0 else 0.0

print("=" * 55)
print(f"📦 Total COGS (180 Days):         ${total_cogs:,.2f}")
print(f"🏷️  Avg Inventory Value:          ${avg_inv_value:,.2f}")
print(f"⚡ Inventory Turnover Ratio:       {turnover_ratio:.2f}x (per 180 days)")
print(f"📅 Days Sales of Inventory (DSI): {dsi:.1f} days")
print("=" * 55)
